# 01a — Capture Schema in Archive Catalogue

Run this to capture the Archive schema so we can compare it against what was provided by NEC, we will compare the live schema against `schema_definition.csv`;

This will help the downstream silver layer as the silver layer requires that the data has references keys.

In [1]:
COMPARED_SCHEMA = "Bronze"
SCHEMA_CSV_PATH = "Files/cfg_files/schema_definition.csv"
FAIL_ON_CRITICAL = True

StatementMeta(, e5930c36-feb0-46af-878d-058d3b5b56ae, 3, Finished, Available, Finished, False)

In [2]:
import re, uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()

def qident(value):
    return "`" + str(value).replace("`", "``") + "`"

def silver_table(schema_name, table_name):
    return f"{SILVER_SCHEMA}.slv_{table_name.lower()}"

def append_rows(table_name, rows, schema):
    if rows:
        spark.createDataFrame(rows, schema).write.format("delta").mode("append").saveAsTable(table_name)

StatementMeta(, e5930c36-feb0-46af-878d-058d3b5b56ae, 4, Finished, Available, Finished, False)

In [3]:
# 1. define target schema
schema_def = StructType([
    StructField("schema_name", StringType(), True),
    StructField("table_name", StringType(), True),
    StructField("ordinal_position", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("data_type", StringType(), True),
    StructField("is_nullable", StringType(), True),
])

schema_records=[]

tables = spark.catalog.listTables(COMPARED_SCHEMA)

for tbl in tables:
    columns= spark.catalog.listColumns(f"{COMPARED_SCHEMA}.{tbl.name}")
    for idx,col in enumerate(columns, start=1):
        schema_records.append((COMPARED_SCHEMA, tbl.name, str(idx), col.name, col.dataType, "true" if col.nullable else "false"))

schema_df=spark.createDataFrame(schema_records, schema=schema_def)   

schema_df.withColumn("contract_loaded_at", F.current_timestamp()) \
        .write.format("delta").mode("append")\
            .saveAsTable("monitoring.cfg_archived_schema_live")





StatementMeta(, e5930c36-feb0-46af-878d-058d3b5b56ae, 5, Finished, Available, Finished, False)

In [4]:
%%sql

SELECT 
    COALESCE(c.schema_name, l.schema_name) AS schema_name,
    COALESCE(c.table_name, l.table_name) AS table_name,
    COALESCE(c.column_name, l.column_name) AS column_name,
    c.data_type AS contract_data_type,
    l.data_type AS live_data_type,
    CASE 
        WHEN c.column_name IS NULL THEN 'UNCONTRACTED_COLUMN_ADDED'
        WHEN l.column_name IS NULL THEN 'CONTRACTED_COLUMN_MISSING'
        ELSE 'MATCH'
    END AS validation_status
FROM monitoring.cfg_schema_contract_column c
FULL OUTER JOIN monitoring.cfg_archived_schema_live l
    ON -- c.schema_name = l.schema_name
   --AND 
   replace(c.table_name,'archived_','') = l.table_name
   AND c.column_name = l.column_name
WHERE c.column_name IS NULL 
   OR l.column_name IS NULL 
   OR c.data_type <> l.data_type;



StatementMeta(, e5930c36-feb0-46af-878d-058d3b5b56ae, 6, Finished, Available, Finished, False)

<Spark SQL result set with 720 rows and 6 fields>

In [5]:
df = spark.read.format("csv").option("header","true").load("Files/wmpp-production-data-export-birmingham/audit/jul26/2026-07-31.csv")
# df now is a Spark DataFrame containing CSV data from "Files/wmpp-production-data-export-birmingham/audit/jul26/2026-07-31.csv".
display(df)

StatementMeta(, e5930c36-feb0-46af-878d-058d3b5b56ae, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 87c33aae-22f1-48aa-8d42-cf61836fa1d9)

In [6]:
df = spark.read.format("csv").option("header","true").load("Files/wmpp-production-data-export-birmingham/latest/provider_submission_docs.csv")
# df now is a Spark DataFrame containing CSV data from "Files/wmpp-production-data-export-birmingham/latest/provider_submission_docs.csv".
display(df)

StatementMeta(, e5930c36-feb0-46af-878d-058d3b5b56ae, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2f33120b-30c0-4713-bf3e-c4c98e13b4d0)